# GeneFlow AI: clasificación taxonómica jerárquica de secuencias de ADN

Este notebook es el cuaderno de trabajo del TFM. Aquí se explora el dataset y, más adelante, se entrenan y evalúan los modelos. **La lógica reutilizable no vive aquí, sino en el paquete `taxonomy_classifier`**, que tiene tests, tipado estricto y un 90 % de cobertura mínima. El notebook solo lo importa y lo usa.

## Requisitos previos

Desde la raíz del repositorio:

| Paso | Comando | Qué hace |
|---|---|---|
| 1 | `uv sync` | Instala el paquete en modo editable junto con los grupos `dev` y `notebooks` (Jupyter, Polars, Matplotlib) |
| 2 | `uv run geneflow prepare` | Descarga SILVA, comprueba los hashes y genera `data/processed/silva_<versión>/dataset.parquet` |
| 3 | `uv run jupyter lab` | Abre Jupyter. Si usas PyCharm o VS Code, selecciona el intérprete `.venv` como kernel |

## Qué hace la celda de configuración

**Recarga automática.** `%autoreload 2` vuelve a cargar el paquete cada vez que ejecutas una celda. Si cambias algo en `src/`, no hace falta reiniciar el kernel.

**Rutas.** `PROJECT_ROOT` se calcula buscando `pyproject.toml` hacia arriba, así que el notebook funciona sea cual sea el directorio desde el que se abra. Todas las demás rutas salen de ahí y de la versión de SILVA:

| Variable | Contenido |
|---|---|
| `RELEASE` | La versión de SILVA en uso (`LATEST_RELEASE` del paquete) |
| `RAW_DIR` | `data/raw/silva_<versión>/`: descargas intactas |
| `PROCESSED_DIR` | `data/processed/silva_<versión>/`: el dataset generado |
| `DATASET_PATH` | El Parquet, con una fila por secuencia |
| `REPORT_PATH` | `build_report.json`: cuántas secuencias se descartaron y por qué, y la configuración de filtros usada |
| `FIGURES_DIR` | `reports/figures/`: figuras exportadas para la memoria (se crea si no existe) |

**Comprobación del dataset.** Si el Parquet no existe, la celda se detiene con un mensaje que indica el comando que hay que ejecutar.

**Datos.**

| Variable | Contenido |
|---|---|
| `dataset` | Un `LazyFrame` de Polars sobre el Parquet. No carga nada en memoria hasta que se llama a `.collect()`, lo que importa con ~856 mil secuencias y ~1,4 GB de texto |
| `report` | El informe de construcción, ya leído como diccionario |
| `RANK_COLUMNS` | Las columnas taxonómicas en orden jerárquico: `domain`, `kingdom`, `phylum`, `class`, `order`, `family`, `genus` |

**Presentación.** Configura el `logging` para ver los mensajes del paquete, cómo muestra Polars las tablas y el estilo de Matplotlib. Las figuras se exportan a 300 ppp, con calidad de impresión.

## Estructura del dataset

| Columna | Tipo | Descripción |
|---|---|---|
| `accession`, `start`, `end` | texto, entero, entero | Identificador de la secuencia en SILVA y coordenadas dentro del registro original |
| `organism` | texto | Nombre del organismo tal como aparece en SILVA. **No es una etiqueta fiable** |
| `sequence` | texto | Secuencia normalizada a ADN (`U` pasa a `T`) en mayúsculas |
| `length`, `n_ambiguous` | entero | Longitud y número de bases ambiguas (distintas de `A`, `C`, `G`, `T`) |
| `lineage` | lista de texto | El linaje completo original, incluidos los rangos no canónicos |
| `domain` … `genus` | texto o nulo | Un rango por columna. Vale nulo si el rango no existe o es de relleno (`Incertae Sedis`, `--other`) |


In [ ]:
%load_ext autoreload
%autoreload 2

import json
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

from taxonomy_classifier.data.build import DATASET_FILENAME, REPORT_FILENAME
from taxonomy_classifier.data.sources import LATEST_RELEASE, RELEASES
from taxonomy_classifier.data.taxonomy import CANONICAL_RANKS

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()
)

RELEASE = RELEASES[LATEST_RELEASE]

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / RELEASE.slug
PROCESSED_DIR = DATA_DIR / "processed" / RELEASE.slug
DATASET_PATH = PROCESSED_DIR / DATASET_FILENAME
REPORT_PATH = PROCESSED_DIR / REPORT_FILENAME
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

RANK_COLUMNS = [rank.value for rank in CANONICAL_RANKS]

if not DATASET_PATH.exists():
    msg = f"{DATASET_PATH} not found; run 'uv run geneflow prepare' from {PROJECT_ROOT}"
    raise FileNotFoundError(msg)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_thousands_separator(".")
pl.Config.set_decimal_separator(",")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "figure.dpi": 110,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "axes.titleweight": "bold",
    }
)

dataset = pl.scan_parquet(DATASET_PATH)
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))